# 02 Newton-Cotes 与复合求积

Newton-Cotes 公式使用等距节点。它们是最直接的插值型求积公式：先用等距节点构造插值多项式，再把插值多项式积分。

本节重点包括中点、梯形、Simpson、Simpson 3/8 公式，以及复合梯形和复合 Simpson 的收敛阶实验。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch04_numerical_integration"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import (
    closed_newton_cotes_weights,
    composite_midpoint,
    composite_simpson,
    composite_trapezoid,
    newton_cotes_integrate,
    simpson_rule,
    trapezoid_rule,
)


## 闭型与开型 Newton-Cotes

闭型 Newton-Cotes 包含端点。例如梯形公式使用 $a,b$，Simpson 公式使用 $a,(a+b)/2,b$。

开型 Newton-Cotes 不使用端点。例如单点开型公式就是中点公式。开型公式在端点函数值不可用或端点有弱奇异时有时更方便。

插值型权重可以由矩条件得到：若标准区间为 $[0,1]$，则希望

$$
\sum_i w_i x_i^k = \int_0^1 x^k\,dx=\frac{1}{k+1},
\qquad k=0,1,\ldots,n.
$$


In [ ]:
def teaching_closed_newton_cotes(order):
    nodes = np.linspace(0.0, 1.0, order + 1)
    vandermonde = np.vander(nodes, order + 1, increasing=True).T
    moments = 1.0 / (np.arange(order + 1) + 1.0)
    weights = np.linalg.solve(vandermonde, moments)
    return nodes, weights

for order in [1, 2, 3]:
    nodes, weights = teaching_closed_newton_cotes(order)
    print(f"闭型 {order} 次公式")
    print("nodes  =", nodes)
    print("weights=", weights)


## 典型单区间公式

中点公式：

$$
M(f)=(b-a)f\left(\frac{a+b}{2}\right).
$$

梯形公式：

$$
T(f)=\frac{b-a}{2}[f(a)+f(b)].
$$

Simpson 公式：

$$
S(f)=\frac{b-a}{6}\left[f(a)+4f\left(\frac{a+b}{2}\right)+f(b)\right].
$$

Simpson 3/8 公式：

$$
S_{3/8}(f)=\frac{3h}{8}\left[f(a)+3f(a+h)+3f(a+2h)+f(b)\right],
\qquad h=\frac{b-a}{3}.
$$

梯形公式的单区间误差含 $f''$，Simpson 公式的单区间误差含 $f^{(4)}$。因此在足够光滑时，Simpson 的全局收敛阶高于梯形。


In [ ]:
def teaching_midpoint(func, a, b):
    return (b - a) * func(0.5 * (a + b))


def teaching_trapezoid(func, a, b):
    return 0.5 * (b - a) * (func(a) + func(b))


def teaching_simpson(func, a, b):
    mid = 0.5 * (a + b)
    return (b - a) * (func(a) + 4.0 * func(mid) + func(b)) / 6.0


def teaching_three_eighths(func, a, b):
    h = (b - a) / 3.0
    return 3.0 * h * (func(a) + 3.0 * func(a + h) + 3.0 * func(a + 2 * h) + func(b)) / 8.0

func = lambda x: x**3
print("积分 x^3, exact=1/4")
for name, rule in [
    ("中点", teaching_midpoint),
    ("梯形", teaching_trapezoid),
    ("Simpson", teaching_simpson),
    ("Simpson 3/8", teaching_three_eighths),
]:
    print(f"{name:12s}", rule(func, 0.0, 1.0))


## 复合公式

高阶 Newton-Cotes 公式可能不稳定，因此实际计算更常把区间切成许多小段，在每段上使用低阶公式。

复合梯形公式：

$$
T_n=h\left[\frac12 f(x_0)+\sum_{i=1}^{n-1}f(x_i)+\frac12 f(x_n)\right].
$$

复合 Simpson 公式要求 $n$ 为偶数：

$$
S_n=\frac{h}{3}\left[f(x_0)+f(x_n)+4\sum_{i=1,3,\ldots,n-1}f(x_i)+2\sum_{i=2,4,\ldots,n-2}f(x_i)\right].
$$


In [ ]:
def teaching_composite_trapezoid(func, a, b, n):
    x = np.linspace(a, b, n + 1)
    y = func(x)
    h = (b - a) / n
    return h * (0.5 * y[0] + y[1:-1].sum() + 0.5 * y[-1])


def teaching_composite_simpson(func, a, b, n):
    if n % 2 != 0:
        raise ValueError("Simpson 复合公式要求 n 为偶数")
    x = np.linspace(a, b, n + 1)
    y = func(x)
    h = (b - a) / n
    return h / 3.0 * (y[0] + y[-1] + 4.0 * y[1:-1:2].sum() + 2.0 * y[2:-1:2].sum())

print("教学版与正式实现对照：")
print(teaching_composite_trapezoid(np.sin, 0.0, math.pi, 16), composite_trapezoid(np.sin, 0.0, math.pi, 16))
print(teaching_composite_simpson(np.sin, 0.0, math.pi, 16), composite_simpson(np.sin, 0.0, math.pi, 16))


## 收敛阶实验

对光滑函数 $f(x)=\sin x$，精确积分为

$$
\int_0^\pi \sin x\,dx=2.
$$

理论上复合梯形为二阶，复合 Simpson 为四阶。下面用步长减半实验验证。


In [ ]:
def observed_orders(errors):
    errors = np.asarray(errors)
    return np.log(errors[:-1] / errors[1:]) / np.log(2.0)

exact = 2.0
n_values = np.array([8, 16, 32, 64, 128, 256])
trap_errors = np.array([abs(composite_trapezoid(np.sin, 0.0, math.pi, int(n)) - exact) for n in n_values])
simp_errors = np.array([abs(composite_simpson(np.sin, 0.0, math.pi, int(n)) - exact) for n in n_values])

print("n    梯形误差        Simpson 误差")
for n, et, es in zip(n_values, trap_errors, simp_errors):
    print(f"{n:<4d} {et: .6e}    {es: .6e}")
print("梯形实验阶:", observed_orders(trap_errors))
print("Simpson 实验阶:", observed_orders(simp_errors))

plt.figure(figsize=(7, 4.2))
plt.loglog(n_values, trap_errors, "o-", label="复合梯形，理论二阶")
plt.loglog(n_values, simp_errors, "s-", label="复合 Simpson，理论四阶")
plt.loglog(n_values, trap_errors[0] * (n_values[0] / n_values) ** 2, "--", color="C0", alpha=0.5, label="二阶参考")
plt.loglog(n_values, simp_errors[0] * (n_values[0] / n_values) ** 4, "--", color="C1", alpha=0.5, label="四阶参考")
plt.xlabel("子区间数 n")
plt.ylabel("绝对误差")
plt.title("复合求积公式的收敛阶")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()


## 高阶 Newton-Cotes 的局限

高阶闭型 Newton-Cotes 公式也可以通过矩条件构造，但节点数增加后，权重可能出现较大的正负交替。负权重不必然错误，但会让公式对函数扰动和舍入误差更敏感。


In [ ]:
orders = [2, 4, 6, 8, 10]
plt.figure(figsize=(7, 4.2))
for order in orders:
    nodes, weights = closed_newton_cotes_weights(order)
    plt.plot(nodes, weights, "o-", label=f"{order} 次")
plt.axhline(0.0, color="black", linewidth=0.8)
plt.xlabel("标准节点")
plt.ylabel("权重")
plt.title("高阶闭型 Newton-Cotes 权重的正负交替")
plt.grid(True, alpha=0.3)
plt.legend(ncol=2)
plt.show()


## 小结

* Newton-Cotes 公式来自等距节点上的插值多项式积分。
* 中点、梯形和 Simpson 是最常用的低阶公式。
* 复合梯形对光滑函数通常呈二阶收敛，复合 Simpson 通常呈四阶收敛。
* 高阶 Newton-Cotes 公式不一定比低阶复合公式稳健，原因之一是权重可能变大并出现正负抵消。
* 当函数局部变化不均匀时，固定等距划分可能浪费函数计算；这会引出 Romberg 和自适应积分。
